In [ ]:
# Adapted from:
#
# "R-hat locker" notebook
# Original version:
# Copyright 2021 Google LLC
# https://github.com/google-research/google-research/tree/master/nested_rhat
#
# Revised version:
# Copyright 2022 Charles C. Margossian
#
# Licensed under the Apache License, Version 2.0
# http://www.apache.org/licenses/LICENSE-2.0
#
# Modifications by Jiaming Chang:
# - ...
# - ...

Objective of this notebook:
Nested R hat window performance on MVN from 2 variables to 100 variables (perhaps 2, 10, 50, 80,100?) and for each of them I can try 3-4 kinds of covariance?

* create MVN distribution of 2, 10, 50, 80, 100 variables, each with AR(1)covariance matrix where $ρ$ ∈ {0.2,0.5,0.7.0.9}
* evaluate the nested r hat performance on each distribution, see where did nested r hat dropped down to 1.01
* compare the nested r hat performance with regular r hat




In [ ]:
!pip install -Uq tfp-nightly[jax]
!pip install inference_gym
!pip install tf-nightly

# remember to restart runtime after changing the environment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.9/390.9 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.3/623.3 MB 905.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 109.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 34.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.38.0 requires proto

In [ ]:
import numpy as np
# from matplotlib.pyplot import *

import os
# in case jax eats up my GPU RAM
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import jax
from jax import random
from jax import numpy as jnp

# from inference_gym import using_jax as gym

from tensorflow_probability.substrates import jax as tfp
tfd = tfp.distributions
# tfb = tfp.bijectors

# from tensorflow_probability.substrates import numpy as tfp_np
# tfd_np = tfp_np.distributions

import matplotlib.pyplot as plt

import pandas as pd

import gc

# check if this is run on a gpu
print(jax.devices())
print(jax.default_backend())

[CudaDevice(id=0)]
gpu


In [ ]:
import psutil

process = psutil.Process(os.getpid())

def mem(msg):
    print(f"{msg}: {process.memory_info().rss / 1024**2:.1f} MB")

Necessary Functions:

In [ ]:
#===================================
# TEST MODULE:
# check if the kernel is correctly set up
#===================================
def dist_checker(target, total_samples_short, initial_state, kernel,biject):
  result_state_short = tfp.mcmc.sample_chain(
          total_samples_short,
          initial_state,
          kernel=kernel,
          seed= random.PRNGKey(2002),
          trace_fn=None
      )
  if (biject):
    result_state_short = target.default_event_space_bijector(result_state_short)
  for i in range(1):
    points = np.array(result_state_short[:, i, :])
    plt.scatter(points[:, 0], points[:, 1], alpha=0.3)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title("test plot")
    plt.show()

# example:
# dist_checker(target, total_samples_short, initial_state, kernel_short, biject=False)

In [ ]:
#===================================
# MCMC Error Module:
# check the MCMC error of the sampling process
#===================================
# this is for later lets focus on the sampling first

In [ ]:
#===================================
# Nested R Hat Module:
# return a list of nested R hat based on customizable window size
#+==================================
def _reduce_variance_interval(x, axis=None, biased=True, keepdims=False):
    # ddof=0 is biased variance (N), ddof=1 is unbiased variance (N-1)
    ddof = 0 if biased else 1
    return jnp.var(x, axis=axis, ddof=ddof, keepdims=keepdims)


def nested_rhat_interval(result_state, num_super_chains,window_size, idx):
  # calculate the nested rhat with "slices" of samples
  num_samples = list(range(0, 1101, window_size))
  nested_rhat_list = []
  for start, end in zip(num_samples[:-1], num_samples[1:]):
    result_state_short = result_state[start:end, :, :]
    # print(f"currently taking samples [{start}:{end}]")
    used_samples = result_state_short.shape[0]
    num_sub_chains = result_state_short.shape[1] // num_super_chains
    num_dimensions = result_state_short.shape[2]

    chain_states = result_state_short.reshape(used_samples, -1, num_sub_chains, num_dimensions)

    mean_chain = jnp.mean(chain_states, axis=0) # estimator from one subchain
    mean_super_chain = jnp.mean(chain_states, axis=(0, 2))  # estimator from one super chain

    variance_chain = _reduce_variance_interval(chain_states, axis=0, biased=False)
    variance_super_chain = _reduce_variance_interval(mean_chain, axis=1, biased=False) \
                           + jnp.mean(variance_chain, axis=1)
                           # sum of variance of mean of one chain (B_k)
                           # and mean of variance of one chain (W_k)

    W = jnp.mean(variance_super_chain, axis=0) # avg of (B_k + W_k)
    B = _reduce_variance_interval(mean_super_chain, axis=0, biased=False) # variance of between super chain

    r_hat = jnp.sqrt((W + B) / W)[idx]
    nested_rhat_list.append(float(r_hat))
  return nested_rhat_list
#===================================
# R Hat Module:
# return a list of R hat based on customizable window size
#+==================================
def compute_rhat(result_state, num_samples, num_warmup = 0):
  return tfp.mcmc.potential_scale_reduction(result_state[num_warmup:num_warmup + num_samples + 1],
                                          independent_chain_ndims = 1).T
#===================================
# R Hat Evaluation Module:
# return a dataframe of nested R hat and traditional R hat
# based on customizable window size
# based on the problem it seems like we need to discard the warmup samples
# for traditional rhat computation, but really?
#+==================================
def rhat_assess(num_simulations,window_size, total_samples_short,
                initial_state,kernel,num_super_chains,idx, biject=False):
  num_samples = list(range(0, 1101, window_size))
  range_iter = list(range(100, total_samples_short+1, window_size))
  interval_names = [
      f"{start}_{end}"
      for start, end in zip(num_samples[:-1], num_samples[1:])
      ]
  nested_rhat = []
  rhat_list = []
  base_key = random.PRNGKey(0)
  keys = random.split(base_key, num_simulations)
  for sim in range(num_simulations):
    # if sim % 10 == 0:
    #     mem(f"Simulation {sim} - start")
    rhat_short = np.empty(len(range_iter))
    result_state_short = tfp.mcmc.sample_chain(
        total_samples_short,
        initial_state,
        kernel=kernel,
        seed=keys[sim],
        trace_fn=None
        )
    # if (sim==0):
    #   print(result_state_short.shape)
    if (biject):
      result_state_short = target.default_event_space_bijector(result_state_short)

    nested_val = nested_rhat_interval(result_state_short, num_super_chains,
                                      window_size,idx)

    for j,n in enumerate(range_iter):
      rhat_short[j] = compute_rhat(result_state_short, n, 100)[0]

    nested_rhat.append(nested_val)
    rhat_list.append(rhat_short)
    # Clean Up:
    jax.block_until_ready(result_state_short)
    # if sim % 10 == 0:
    #     mem(f"Simulation {sim} - after sample")
    del result_state_short
    del rhat_short
    del nested_val
    if sim % 10 == 0:
      gc.collect()
      # jax.clear_caches()

    # if sim % 10 == 0:
    #     mem(f"Simulation {sim} - after delete")

  nested_df = pd.DataFrame(nested_rhat, columns=interval_names)
  rhat_df = pd.DataFrame(rhat_list, columns=range_iter)

  return nested_df, rhat_df

In [ ]:
#===================================
# MVN Module:
# return the target MVN distribution, kernel for sampling such
# distribution, initial state and total number of samples
#+==================================

def MVN_builder(num_dim, rho, num_superChain, num_subChain):
  def target_log_prob_fn(x):
    return target.log_prob(x)
  def initialize(shape, key=random.PRNGKey(123)):
    return random.normal(key, shape + (num_dim,))

  # mean array:
  mu = [float(0) for i in range(num_dim)]
  # covariance matrix:
  cov = [ [rho**(abs(i-j)) for j in range(num_dim)] for i in range(num_dim)]
  target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=jnp.array(cov))

  init_step_size = 0.5
  num_warmup_short, num_sampling_short = 100, 1000
  total_samples_short = num_warmup_short + num_sampling_short

  kernel_short = tfp.mcmc.HamiltonianMonteCarlo(target_log_prob_fn,init_step_size,1)
  kernel_short = tfp.experimental.mcmc.GradientBasedTrajectoryLengthAdaptation(kernel_short,num_warmup_short)
  kernel_short = tfp.mcmc.DualAveragingStepSizeAdaptation(
      kernel_short,
      num_warmup_short,
      target_accept_prob=0.75,
      reduce_fn=tfp.math.reduce_log_harmonic_mean_exp)

  initial_state = initialize((num_superChain,))
  initial_state = np.repeat(
      initial_state,
      num_subChain,
      axis=0
  )
  return target, kernel_short, initial_state, total_samples_short

In [ ]:
# One example:
# num_dim = 2
# rho = 0.7
# num_superChain = 4
# num_subChain = 128 # in total 512 chains, each superchain has 128 subchains

# target, kernel, initial_state, total_samples = MVN_builder(num_dim, rho, num_superChain, num_subChain)
# num_simulations = 100
# window_size = 20
# df_nested, df_rhat = rhat_assess(num_simulations,window_size,
#                              total_samples, initial_state,
#                              kernel, num_superChain,0)

In [ ]:
#===================================
# Plotting Module: obtain the plots, just plots??
# input: two dataframes, one for nested rhat and one for regular rhat
# this module should output the mean of r hat and nested r hat for every interval,
# and compare the trend
#===================================
def mean_compare(df_nested, df_rhat,plot=False):

    threshold = 1.01
    nested_mean = df_nested.mean()
    rhat_mean = df_rhat.mean()

    nested_x = [int(c.split('_')[1]) for c in df_nested.columns]
    rhat_x = list(df_rhat.columns)

    nested_idx = np.where(nested_mean.values <= threshold)[0]
    nested_cross = nested_x[nested_idx[0]]
    rhat_idx = np.where(rhat_mean.values <= threshold)[0]
    if len(rhat_idx) > 0:
      rhat_cross = rhat_x[rhat_idx[0]]
    else:
      print(f"the least mean R hat value is {min(rhat_mean.values)}")
      rhat_cross = -1

    if plot:
      fig, ax = plt.subplots(figsize=(20, 9))
      ax.plot(
          nested_x,
          nested_mean.values,
          marker='o',
          label='Nested R-hat'
      )
      ax.plot(
          rhat_x,
          rhat_mean.values,
          marker='s',
          label='R-hat'
      )

      if len(nested_idx) > 0:
          ax.axvline(
              nested_cross,
              color='C0',
              linestyle=':',
              alpha=0.8,
              label=f'Nested ≤ {threshold} ({nested_cross})'
          )

      if len(rhat_idx) > 0:
          ax.axvline(
              rhat_cross,
              color='C1',
              linestyle=':',
              alpha=0.8,
              label=f'R-hat ≤ {threshold} ({rhat_cross})'
          )

      ax.axhline(threshold, color='red', linestyle='--')

      ax.set_xlabel('Number of samples')
      ax.set_ylabel('Mean diagnostic value')
      ax.set_title('Nested R-hat vs R-hat')
      ax.legend()

      plt.show()
    return nested_cross, rhat_cross
# example:
# mean_compare(df_nested, df_rhat)

MVN set up:
We will set up several MVN distributions:  
MVN with 2, 10, 50, 75, 100 variables,   
and for each of them, the covariance is an AR(1) covariance matrix with tunable $ρ$

In [ ]:
#
dim_list = [2,10,50,75,100]
rho_list = [0.2,0.5,0.7,0.9]
# dim_list = [2]
# rho_list = [0.2]
num_simulations = 100
window_size = 20

for num_dim in dim_list:
  for rho in rho_list:
    mem(f"Simulation: Dimention:{num_dim}, rho:{rho} - start")
    num_superChain = 4
    num_subChain = 128

    target,kernel,initial_state, total_samples = MVN_builder(num_dim, rho, num_superChain, num_subChain)
    df_nested, df_rhat = rhat_assess(num_simulations,window_size,
                                     total_samples, initial_state,
                                     kernel, num_superChain,0)
    nested_idx, rhat_idx = mean_compare(df_nested, df_rhat, plot=False)
    print(f"num_dim: {num_dim}, rho: {rho}, nested_idx: {nested_idx}, rhat_idx: {rhat_idx}")
    # stub for dataframe construction
    del target
    del kernel
    del initial_state
    del df_nested
    del df_rhat

    gc.collect()
    jax.clear_caches()
# num_dim: 2, rho: 0.2, nested_idx: 40, rhat_idx: 142
# num_dim: 2, rho: 0.5, nested_idx: 40, rhat_idx: 182
# num_dim: 2, rho: 0.7, nested_idx: 40, rhat_idx: 242
# num_dim: 2, rho: 0.9, nested_idx: 40, rhat_idx: 342
# num_dim: 10, rho: 0.2, nested_idx: 40, rhat_idx: 202
# num_dim: 10, rho: 0.5, nested_idx: 40, rhat_idx: 222
# num_dim: 10, rho: 0.7, nested_idx: 60, rhat_idx: 302
# num_dim: 10, rho: 0.9, nested_idx: 60, rhat_idx: 402
# num_dim: 50, rho: 0.2, nested_idx: 40, rhat_idx: 282
# num_dim: 50, rho: 0.5, nested_idx: 60, rhat_idx: 282
# num_dim: 50, rho: 0.7, nested_idx: 60, rhat_idx: 342
# num_dim: 50, rho: 0.9, nested_idx: 80, rhat_idx: 762
# num_dim: 75, rho: 0.2, nested_idx: 40, rhat_idx: 302
# num_dim: 75, rho: 0.5, nested_idx: 60, rhat_idx: 302
# num_dim: 75, rho: 0.7, nested_idx: 60, rhat_idx: 382

##############second time##################
'''
Simulation: Dimention:2, rho:0.2 - start: 1244.1 MB
num_dim: 2, rho: 0.2, nested_idx: 40, rhat_idx: 100
Simulation: Dimention:2, rho:0.5 - start: 3117.8 MB
num_dim: 2, rho: 0.5, nested_idx: 40, rhat_idx: 100
Simulation: Dimention:2, rho:0.7 - start: 3121.4 MB
num_dim: 2, rho: 0.7, nested_idx: 40, rhat_idx: 100
Simulation: Dimention:2, rho:0.9 - start: 3121.6 MB
num_dim: 2, rho: 0.9, nested_idx: 40, rhat_idx: 120
Simulation: Dimention:10, rho:0.2 - start: 3121.8 MB
num_dim: 10, rho: 0.2, nested_idx: 40, rhat_idx: 100
Simulation: Dimention:10, rho:0.5 - start: 3158.7 MB
num_dim: 10, rho: 0.5, nested_idx: 40, rhat_idx: 100
Simulation: Dimention:10, rho:0.7 - start: 3159.1 MB
num_dim: 10, rho: 0.7, nested_idx: 60, rhat_idx: 100
Simulation: Dimention:10, rho:0.9 - start: 3159.4 MB
num_dim: 10, rho: 0.9, nested_idx: 60, rhat_idx: 180
Simulation: Dimention:50, rho:0.2 - start: 3159.5 MB
num_dim: 50, rho: 0.2, nested_idx: 40, rhat_idx: 100
Simulation: Dimention:50, rho:0.5 - start: 3175.8 MB
num_dim: 50, rho: 0.5, nested_idx: 60, rhat_idx: 100
Simulation: Dimention:50, rho:0.7 - start: 3178.0 MB
num_dim: 50, rho: 0.7, nested_idx: 60, rhat_idx: 100
Simulation: Dimention:50, rho:0.9 - start: 3178.2 MB
num_dim: 50, rho: 0.9, nested_idx: 80, rhat_idx: 540
Simulation: Dimention:75, rho:0.2 - start: 3178.4 MB
num_dim: 75, rho: 0.2, nested_idx: 40, rhat_idx: 120
Simulation: Dimention:75, rho:0.5 - start: 3191.9 MB
num_dim: 75, rho: 0.5, nested_idx: 60, rhat_idx: 100
Simulation: Dimention:75, rho:0.7 - start: 3192.3 MB
num_dim: 75, rho: 0.7, nested_idx: 60, rhat_idx: 100
Simulation: Dimention:75, rho:0.9 - start: 3192.6 MB
num_dim: 75, rho: 0.9, nested_idx: 100, rhat_idx: -1
Simulation: Dimention:100, rho:0.2 - start: 3192.8 MB
num_dim: 100, rho: 0.2, nested_idx: 40, rhat_idx: 120
Simulation: Dimention:100, rho:0.5 - start: 3227.4 MB
num_dim: 100, rho: 0.5, nested_idx: 40, rhat_idx: 100
Simulation: Dimention:100, rho:0.7 - start: 3227.9 MB
num_dim: 100, rho: 0.7, nested_idx: 40, rhat_idx: 100
Simulation: Dimention:100, rho:0.9 - start: 3227.9 MB
num_dim: 100, rho: 0.9, nested_idx: 80, rhat_idx: -1
'''

Simulation: Dimention:2, rho:0.2 - start: 1244.1 MB
num_dim: 2, rho: 0.2, nested_idx: 40, rhat_idx: 100
Simulation: Dimention:2, rho:0.5 - start: 3117.8 MB
num_dim: 2, rho: 0.5, nested_idx: 40, rhat_idx: 100
Simulation: Dimention:2, rho:0.7 - start: 3121.4 MB
num_dim: 2, rho: 0.7, nested_idx: 40, rhat_idx: 100
Simulation: Dimention:2, rho:0.9 - start: 3121.6 MB
num_dim: 2, rho: 0.9, nested_idx: 40, rhat_idx: 120
Simulation: Dimention:10, rho:0.2 - start: 3121.8 MB
num_dim: 10, rho: 0.2, nested_idx: 40, rhat_idx: 100
Simulation: Dimention:10, rho:0.5 - start: 3158.7 MB
num_dim: 10, rho: 0.5, nested_idx: 40, rhat_idx: 100
Simulation: Dimention:10, rho:0.7 - start: 3159.1 MB
num_dim: 10, rho: 0.7, nested_idx: 60, rhat_idx: 100
Simulation: Dimention:10, rho:0.9 - start: 3159.4 MB
num_dim: 10, rho: 0.9, nested_idx: 60, rhat_idx: 180
Simulation: Dimention:50, rho:0.2 - start: 3159.5 MB
num_dim: 50, rho: 0.2, nested_idx: 40, rhat_idx: 100
Simulation: Dimention:50, rho:0.5 - start: 3175.8 MB
n

we can see that when the dimension comes to 75 and 100, and the covariance rho is 0.9, the average traditional r hat could not reach 1.01 in 1100 iterations

In [ ]:
# a special cade where the mean of traditional r hat cannot drop below 1.01 after 1100 iterations
dim_list = [100]
rho_list = [0.9]

for num_dim in dim_list:
  for rho in rho_list:
    mem(f"Simulation: Dimention:{num_dim}, rho:{rho} - start")
    num_superChain = 4
    num_subChain = 128

    target,kernel,initial_state, total_samples = MVN_builder(num_dim, rho, num_superChain, num_subChain)
    df_nested, df_rhat = rhat_assess(num_simulations,window_size,
                                     total_samples, initial_state,
                                     kernel, num_superChain,0)
    nested_idx, rhat_idx = mean_compare(df_nested, df_rhat, plot=False)
    print(f"num_dim: {num_dim}, rho: {rho}, nested_idx: {nested_idx}, rhat_idx: {rhat_idx}")
    # stub for dataframe construction
    del target
    del kernel
    del initial_state
    del df_nested
    del df_rhat

    gc.collect()
    jax.clear_caches()

Simulation: Dimention:100, rho:0.9 - start: 3228.3 MB
the least mean R hat value is 1.0162427914142609
num_dim: 100, rho: 0.9, nested_idx: 80, rhat_idx: -1
